In [1]:
# Block 1: Setup, Imports, and Data Loading
# ==============================================================================
import warnings
import time
import pandas as pd
import numpy as np
import gc

# Model Imports
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

# Sklearn Imports
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.cluster import MiniBatchKMeans

# Ignore warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

# --- Load Data ---
try:
    # We drop the low-variance columns they identified right away
    drop_cols=['id', 'golf', 'view_rainier', 'view_skyline',
               'view_lakesamm', 'view_otherwater', 'view_other']
    DATA_PATH = './'
    df_train = pd.read_csv(DATA_PATH + 'dataset.csv').drop(columns=drop_cols)
    df_test = pd.read_csv(DATA_PATH + 'test.csv').drop(columns=drop_cols)
    print("Raw data loaded successfully.")
except FileNotFoundError:
    print("ERROR: Could not find 'dataset.csv' or 'test.csv'.")
    exit()

# --- Global Constants & Target Variable ---
N_SPLITS = 5
RANDOM_STATE = 42
COMPETITION_ALPHA = 0.1

y_true = df_train['sale_price'].copy()
# We keep sale_price in df_train for feature engineering
print("Setup complete.")

Libraries imported successfully.
Raw data loaded successfully.
Setup complete.


In [2]:
# Block 2: Advanced Feature Engineering (Corrected)
# ==============================================================================
print("--- Starting Block 2: Advanced Feature Engineering ---")

def feature_engineer(df):
    """Creates a rich set of features from the raw data."""
    # --- A) Brute-Force Numerical Interactions ---
    NUMS = ['area', 'land_val', 'imp_val', 'sqft_lot', 'sqft', 'sqft_1', 'grade']
    for i in range(len(NUMS)):
        for j in range(i + 1, len(NUMS)):
            df[f'{NUMS[i]}_x_{NUMS[j]}'] = df[NUMS[i]] * df[NUMS[j]]

    # --- B) Date Features ---
    df['sale_date'] = pd.to_datetime(df['sale_date'])
    df['year'] = df['sale_date'].dt.year
    df['month'] = df['sale_date'].dt.month
    df['dayofyear'] = df['sale_date'].dt.dayofyear
    df['year_diff'] = df['year'] - df['year_built']
    df['year_reno_diff'] = df['year'] - df['year_reno'].replace(0, np.nan)

    # --- C) Geospatial Features ---
    coords = df[['latitude', 'longitude']].copy()
    kmeans = MiniBatchKMeans(n_clusters=15, random_state=RANDOM_STATE, batch_size=256, n_init=10)
    df['geo_cluster'] = kmeans.fit_predict(coords)

    # --- D) Log transform ---
    for c in ['land_val_x_imp_val', 'land_val_x_sqft', 'imp_val_x_sqft']:
        if c in df.columns:
            df[c] = np.log1p(df[c]) # fillna is handled later

    return df

# Combine data for consistent processing
train_ids = df_train.index
test_ids = df_test.index
df_train['is_train'] = 1
df_test['is_train'] = 0
all_data = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)

# Apply Feature Engineering
all_data = feature_engineer(all_data)

# Final Cleanup
cols_to_drop_final = ['sale_date']
all_data = all_data.drop(columns=cols_to_drop_final)

# === FIX: Intelligent Imputation ===
# Identify text columns for special handling
text_cols_impute = ['subdivision', 'zoning', 'city', 'sale_warning', 'join_status', 'submarket']
for col in all_data.columns:
    if col in text_cols_impute:
        # Fill missing text with a placeholder
        all_data[col] = all_data[col].fillna('missing')
    elif all_data[col].dtype == 'object':
         # Handle other potential object columns if any
         all_data[col] = all_data[col].fillna('missing')
    else:
        # Fill missing numeric values with 0
        all_data[col] = all_data[col].fillna(0)

# Separate back into X and X_test
X = all_data[all_data['is_train'] == 1].drop(columns=['is_train', 'sale_price'])
X_test = all_data[all_data['is_train'] == 0].drop(columns=['is_train', 'sale_price'])

# Restore original index
X.index = train_ids
X_test.index = test_ids

print(f"\nSynthesized FE complete. Total features: {X.shape[1]}")
gc.collect()

--- Starting Block 2: Advanced Feature Engineering ---

Synthesized FE complete. Total features: 65


20

In [6]:
# Block 3: Stage 1 - K-Fold Training of Mean Model Ensemble (Workaround for Old XGBoost)
# ==============================================================================
print("\n--- STAGE 1: K-Fold Training of Mean Model Ensemble ---")

# (Imports and setup remain the same)
grade_for_stratify = pd.read_csv(DATA_PATH + 'dataset.csv')['grade']
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
oof_mean_preds, test_mean_preds = np.zeros(len(X)), np.zeros(len(X_test))
text_cols = ['subdivision', 'zoning', 'city', 'sale_warning', 'join_status', 'submarket']
categorical_features = text_cols + ['geo_cluster']
xgb_params = {'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'eta': 0.02, 'max_depth': 7, 'subsample': 0.8, 'colsample_bytree': 0.7, 'random_state': RANDOM_STATE, 'n_jobs': -1}
lgb_params = {'objective': 'regression_l1', 'metric': 'rmse', 'n_estimators': 2000, 'learning_rate': 0.01, 'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 1, 'lambda_l1': 0.1, 'lambda_l2': 0.1, 'num_leaves': 31, 'verbose': -1, 'n_jobs': -1, 'seed': RANDOM_STATE}
cb_params = {'iterations': 2000, 'learning_rate': 0.02, 'depth': 7, 'loss_function': 'RMSE', 'eval_metric': 'RMSE', 'random_seed': RANDOM_STATE, 'verbose': 0, 'cat_features': categorical_features}

for fold, (train_idx, val_idx) in enumerate(skf.split(X, grade_for_stratify)):
    print(f"\n--- Fold {fold+1}/{N_SPLITS} ---")
    # (Data prep for the fold remains the same)
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y_true.iloc[train_idx], y_true.iloc[val_idx]
    X_train_fold, X_val_fold, X_test_fold = X_train.copy(), X_val.copy(), X_test.copy()
    for col in text_cols:
        tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=150)
        X_train_fold[col] = X_train_fold[col].astype(str)
        X_val_fold[col] = X_val_fold[col].astype(str)
        X_test_fold[col] = X_test_fold[col].astype(str)
        tfidf.fit(X_train_fold[col])
        train_tfidf_matrix = tfidf.transform(X_train_fold[col]).toarray()
        val_tfidf_matrix = tfidf.transform(X_val_fold[col]).toarray()
        test_tfidf_matrix = tfidf.transform(X_test_fold[col]).toarray()
        num_features = train_tfidf_matrix.shape[1]
        train_tfidf = pd.DataFrame(train_tfidf_matrix, index=X_train_fold.index, columns=[f'{col}_tfidf_{i}' for i in range(num_features)])
        val_tfidf = pd.DataFrame(val_tfidf_matrix, index=X_val_fold.index, columns=[f'{col}_tfidf_{i}' for i in range(num_features)])
        test_tfidf = pd.DataFrame(test_tfidf_matrix, index=X_test_fold.index, columns=[f'{col}_tfidf_{i}' for i in range(num_features)])
        X_train_fold = pd.concat([X_train_fold, train_tfidf], axis=1)
        X_val_fold = pd.concat([X_val_fold, val_tfidf], axis=1)
        X_test_fold = pd.concat([X_test_fold, test_tfidf], axis=1)
    X_train_fold = X_train_fold.drop(columns=text_cols)
    X_val_fold = X_val_fold.drop(columns=text_cols)
    X_test_fold = X_test_fold.drop(columns=text_cols)

    # --- Train Models ---
    # XGBoost
    model_xgb = xgb.XGBRegressor(**xgb_params, n_estimators=2500)
    
    # --- WORKAROUND: Remove all early stopping arguments from .fit() ---
    # This will train for the full n_estimators.
    model_xgb.fit(X_train_fold, y_train, eval_set=[(X_val_fold, y_val)], verbose=False)
    
    preds_xgb_val = model_xgb.predict(X_val_fold)
    preds_xgb_test = model_xgb.predict(X_test_fold)

    # LightGBM (Correct)
    model_lgb = lgb.LGBMRegressor(**lgb_params)
    model_lgb.fit(X_train_fold, y_train, eval_set=[(X_val_fold, y_val)], callbacks=[lgb.early_stopping(100, verbose=False)])
    preds_lgb_val = model_lgb.predict(X_val_fold)
    preds_lgb_test = model_lgb.predict(X_test_fold)

    # CatBoost (Correct)
    X_train_cb = X_train.copy(); X_val_cb = X_val.copy(); X_test_cb = X_test.copy()
    for col in categorical_features:
        X_train_cb[col] = X_train_cb[col].astype(str)
        X_val_cb[col] = X_val_cb[col].astype(str)
        X_test_cb[col] = X_test_cb[col].astype(str)
    model_cb = cb.CatBoostRegressor(**cb_params)
    model_cb.fit(X_train_cb, y_train, eval_set=[(X_val_cb, y_val)])
    preds_cb_val = model_cb.predict(X_val_cb)
    preds_cb_test = model_cb.predict(X_test_cb)

    # Ensemble Predictions
    fold_oof_preds = (preds_xgb_val + preds_lgb_val + preds_cb_val) / 3
    fold_test_preds = (preds_xgb_test + preds_lgb_test + preds_cb_test) / 3

    oof_mean_preds[val_idx] = fold_oof_preds
    test_mean_preds += fold_test_preds / N_SPLITS
    
    print(f"Fold {fold+1} OOF RMSE: {np.sqrt(mean_squared_error(y_val, fold_oof_preds)):,.2f}")
    gc.collect()

final_mean_rmse = np.sqrt(mean_squared_error(y_true, oof_mean_preds))
print(f"\n# Stage 1 (Mean) Final OOF RMSE: ${final_mean_rmse:,.2f}")


--- STAGE 1: K-Fold Training of Mean Model Ensemble ---

--- Fold 1/5 ---
Fold 1 OOF RMSE: 104,222.16

--- Fold 2/5 ---
Fold 2 OOF RMSE: 103,385.78

--- Fold 3/5 ---
Fold 3 OOF RMSE: 103,454.21

--- Fold 4/5 ---
Fold 4 OOF RMSE: 103,222.99

--- Fold 5/5 ---
Fold 5 OOF RMSE: 103,707.02

# Stage 1 (Mean) Final OOF RMSE: $103,599.02


In [9]:
# Block 4: Stage 2 - K-Fold Training of Error Model Ensemble (Corrected)
# ==============================================================================
print("\n--- STAGE 2: K-Fold Training of Error Model Ensemble ---")

# New target is the absolute error from the mean model
error_target = np.abs(y_true - oof_mean_preds)

# Add OOF mean predictions as a feature
X_for_error = X.copy()
X_for_error['mean_pred_oof'] = oof_mean_preds
X_test_for_error = X_test.copy()
X_test_for_error['mean_pred_oof'] = test_mean_preds

# Initialize prediction arrays
oof_error_preds = np.zeros(len(X))
test_error_preds = np.zeros(len(X_test))

# === FIX: Create separate, modified parameters for Stage 2 ===
lgb_params_error = lgb_params.copy()
lgb_params_error['n_estimators'] = 1500

cb_params_error = cb_params.copy()
cb_params_error['iterations'] = 1500

xgb_params_error = xgb_params.copy() # No changes needed but good practice

# Re-use the same folds for consistency
for fold, (train_idx, val_idx) in enumerate(skf.split(X, grade_for_stratify)):
    print(f"\n--- Fold {fold+1}/{N_SPLITS} ---")
    X_train, X_val = X_for_error.iloc[train_idx], X_for_error.iloc[val_idx]
    y_train, y_val = error_target.iloc[train_idx], error_target.iloc[val_idx]

    # --- TF-IDF Features ---
    X_train_fold, X_val_fold, X_test_fold = X_train.copy(), X_val.copy(), X_test_for_error.copy()
    for col in text_cols:
        tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=150)
        X_train_fold[col], X_val_fold[col], X_test_fold[col] = (X_train_fold[col].astype(str), X_val_fold[col].astype(str), X_test_fold[col].astype(str))
        tfidf.fit(X_train_fold[col])
        train_tfidf_matrix, val_tfidf_matrix, test_tfidf_matrix = (tfidf.transform(X_train_fold[col]).toarray(), tfidf.transform(X_val_fold[col]).toarray(), tfidf.transform(X_test_fold[col]).toarray())
        num_features = train_tfidf_matrix.shape[1]
        train_tfidf, val_tfidf, test_tfidf = (pd.DataFrame(train_tfidf_matrix, index=X_train_fold.index, columns=[f'{col}_tfidf_{i}' for i in range(num_features)]), pd.DataFrame(val_tfidf_matrix, index=X_val_fold.index, columns=[f'{col}_tfidf_{i}' for i in range(num_features)]), pd.DataFrame(test_tfidf_matrix, index=X_test_fold.index, columns=[f'{col}_tfidf_{i}' for i in range(num_features)]))
        X_train_fold, X_val_fold, X_test_fold = (pd.concat([X_train_fold, train_tfidf], axis=1), pd.concat([X_val_fold, val_tfidf], axis=1), pd.concat([X_test_fold, test_tfidf], axis=1))
    X_train_fold, X_val_fold, X_test_fold = (X_train_fold.drop(columns=text_cols), X_val_fold.drop(columns=text_cols), X_test_fold.drop(columns=text_cols))
    
    # --- Train Models ---
    # XGBoost (Native API)
    dtrain, dval, dtest = (xgb.DMatrix(X_train_fold, label=y_train), xgb.DMatrix(X_val_fold, label=y_val), xgb.DMatrix(X_test_fold))
    model_xgb = xgb.train(params=xgb_params_error, dtrain=dtrain, num_boost_round=1500, evals=[(dval, 'eval')], early_stopping_rounds=75, verbose_eval=False)
    preds_xgb_val = model_xgb.predict(dval, iteration_range=(0, model_xgb.best_iteration))
    preds_xgb_test = model_xgb.predict(dtest, iteration_range=(0, model_xgb.best_iteration))

    # LightGBM
    model_lgb = lgb.LGBMRegressor(**lgb_params_error) # Use the new params
    model_lgb.fit(X_train_fold, y_train, eval_set=[(X_val_fold, y_val)], callbacks=[lgb.early_stopping(75, verbose=False)])
    preds_lgb_val = model_lgb.predict(X_val_fold)
    preds_lgb_test = model_lgb.predict(X_test_fold)

    # CatBoost for Error
    X_train_cb, X_val_cb, X_test_cb = (X_train.copy(), X_val.copy(), X_test_for_error.copy())
    for col in categorical_features:
        X_train_cb[col], X_val_cb[col], X_test_cb[col] = (X_train_cb[col].astype(str), X_val_cb[col].astype(str), X_test_cb[col].astype(str))
    model_cb = cb.CatBoostRegressor(**cb_params_error) # Use the new params
    model_cb.fit(X_train_cb, y_train, eval_set=[(X_val_cb, y_val)])
    preds_cb_val = model_cb.predict(X_val_cb)
    preds_cb_test = model_cb.predict(X_test_cb)

    # --- Ensemble Predictions ---
    fold_oof_preds = (preds_xgb_val + preds_lgb_val + preds_cb_val) / 3
    fold_test_preds = (preds_xgb_test + preds_lgb_test + preds_cb_test) / 3
    oof_error_preds[val_idx] = fold_oof_preds
    test_error_preds += fold_test_preds / N_SPLITS

    print(f"Fold {fold+1} OOF Error RMSE: {np.sqrt(mean_squared_error(y_val, fold_oof_preds)):,.2f}")
    gc.collect()

final_error_rmse = np.sqrt(mean_squared_error(error_target, oof_error_preds))
print(f"\n# Stage 2 (Error) Final OOF RMSE: ${final_error_rmse:,.2f}")


--- STAGE 2: K-Fold Training of Error Model Ensemble ---

--- Fold 1/5 ---
Fold 1 OOF Error RMSE: 66,780.99

--- Fold 2/5 ---
Fold 2 OOF Error RMSE: 65,288.92

--- Fold 3/5 ---
Fold 3 OOF Error RMSE: 66,238.69

--- Fold 4/5 ---
Fold 4 OOF Error RMSE: 65,655.65

--- Fold 5/5 ---
Fold 5 OOF Error RMSE: 66,700.53

# Stage 2 (Error) Final OOF RMSE: $66,135.52


In [10]:
# Block 5: Final Asymmetric Calibration and Submission
# ==============================================================================
print("\n--- Final Asymmetric Calibration ---")

def winkler_score(y_true, lower, upper, alpha=0.1, return_coverage=False):
    width = upper - lower
    penalty_lower = np.where(y_true < lower, (2 / alpha) * (lower - y_true), 0)
    penalty_upper = np.where(y_true > upper, (2 / alpha) * (y_true - upper), 0)
    score = width + penalty_lower + penalty_upper
    if return_coverage:
        coverage = np.mean((y_true >= lower) & (y_true <= upper))
        return np.mean(score), coverage
    return np.mean(score)

# Clip the OOF error predictions to be non-negative
oof_error_final = np.clip(oof_error_preds, 0, None)

best_a, best_b, best_metric = 2.0, 2.0, float('inf')

print("Starting grid search for optimal multipliers...")
# Define a search space based on previous results
search_space_a = np.arange(1.80, 2.21, 0.01)
search_space_b = np.arange(2.00, 2.41, 0.01)

for a in search_space_a:
    for b in search_space_b:
        low = oof_mean_preds - oof_error_final * a
        high = oof_mean_preds + oof_error_final * b
        
        metric, coverage = winkler_score(y_true, low, high, alpha=COMPETITION_ALPHA, return_coverage=True)
        
        if metric < best_metric:
            best_metric = metric
            best_a, best_b = a, b
            print(f"New best score: {best_metric:,.2f} | Coverage: {coverage:.3f} | a={best_a:.2f}, b={best_b:.2f}")

print(f"\nGrid search complete. Final OOF Score: {best_metric:,.2f}. Best multipliers: a={best_a:.2f}, b={best_b:.2f}")

# --- Create Final Submission ---
print("\nCreating final submission file...")
test_error_final = np.clip(test_error_preds, 0, None)

final_lower = test_mean_preds - test_error_final * best_a
final_upper = test_mean_preds + test_error_final * best_b
final_upper = np.maximum(final_lower, final_upper) # Ensure upper is always > lower

# Reload original test set to get the 'id' column
submission_df = pd.read_csv(DATA_PATH + 'test.csv', usecols=['id'])
submission_df['pi_lower'] = final_lower
submission_df['pi_upper'] = final_upper

submission_df.to_csv('submission_supercharged_v1.csv', index=False)
print("\n'submission_supercharged_v1.csv' created successfully!")
display(submission_df.head())


--- Final Asymmetric Calibration ---
Starting grid search for optimal multipliers...
New best score: 325,743.09 | Coverage: 0.851 | a=1.80, b=2.00
New best score: 325,452.63 | Coverage: 0.852 | a=1.80, b=2.01
New best score: 325,171.73 | Coverage: 0.852 | a=1.80, b=2.02
New best score: 324,901.10 | Coverage: 0.853 | a=1.80, b=2.03
New best score: 324,638.84 | Coverage: 0.854 | a=1.80, b=2.04
New best score: 324,385.38 | Coverage: 0.855 | a=1.80, b=2.05
New best score: 324,140.15 | Coverage: 0.856 | a=1.80, b=2.06
New best score: 323,902.75 | Coverage: 0.856 | a=1.80, b=2.07
New best score: 323,673.61 | Coverage: 0.857 | a=1.80, b=2.08
New best score: 323,453.39 | Coverage: 0.858 | a=1.80, b=2.09
New best score: 323,241.42 | Coverage: 0.859 | a=1.80, b=2.10
New best score: 323,036.92 | Coverage: 0.860 | a=1.80, b=2.11
New best score: 322,840.29 | Coverage: 0.860 | a=1.80, b=2.12
New best score: 322,652.56 | Coverage: 0.861 | a=1.80, b=2.13
New best score: 322,471.91 | Coverage: 0.862 |

,id,pi_lower,pi_upper
0,200000,824224.799242,1.101544e+06
1,200001,560038.315716,7.867216e+05
2,200002,442902.841882,6.641397e+05
3,200003,309832.693113,4.453248e+05
4,200004,312397.019690,8.142707e+05
